## Imports

In [0]:
from pyspark.sql.functions import col, lower, lit, current_timestamp
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
import boto3

## Read Parameters

In [0]:

spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix


In [0]:
env = get_env_prefix()
env = 'p'
print(env)

In [0]:
Businessdate = dbutils.widgets.get("Businessdate")

if not Businessdate:
    PreviousBusinessDay = spark.sql(f"""                    
                            select PreviousBusinessDay
                            from `{env}-cdh`.reference.ref_calendar
                            where CalendarDate = cast(getdate() as date)
                        """).first()[0]
    Businessdate = spark.sql(f"""                    
                            select PreviousBusinessDay
                            from `{env}-cdh`.reference.ref_calendar
                            where CalendarDate = '{PreviousBusinessDay}'
                        """).first()[0]

## Read Tables

In [0]:
fs_cases_snap = spark.sql(f"""
                                select distinct 
                                        case_code
                                      , cust_code
                                      , extc_code_primary
                                      , emp_tag
                                      , case_tag
                                      , QUE_CODE
                                      , SPET_CODE
                                from `{env}-cdh`.landed.qca_fs_cases_snap
                                where BusinessDate = '{Businessdate}'
                            """)

fs_cases_snap.createOrReplaceTempView("fs_cases_snap")

In [0]:
dim_cases_static = spark.sql(f"""
                                select distinct 
                                        case_code
                                      , case_account_for_search
                                      , CASE_CODE_PRIMARY
                                from `{env}-cdh`.landed.qca_dim_cases_static
                                where BusinessDate = '{Businessdate}'
                            """)

dim_cases_static.createOrReplaceTempView("dim_cases_static")

In [0]:
dim_external_company = spark.sql(f"""
                                    select distinct 
                                           EXTC_ID
                                         , EXTC_NAME
                                    from `{env}-cdh`.landed.qca_dim_external_company
                                    where BusinessDate = '{Businessdate}'
                                """)

dim_external_company.createOrReplaceTempView("dim_external_company")

In [0]:
dim_branch = spark.sql(f"""
                            select distinct 
                                    BRN_ID
                                  , BRN_INTCODE
                                  , BRN_DESC
                            from `{env}-cdh`.landed.qca_dim_branch
                            where BusinessDate = '{Businessdate}'
                        """)

dim_branch.createOrReplaceTempView("dim_branch")

In [0]:
dim_queue = spark.sql(f"""
                        select distinct 
                                QUE_ID
                              , QUE_DESC
                        from `{env}-cdh`.landed.qca_dim_queue
                        where BusinessDate = '{Businessdate}'
                    """)

dim_queue.createOrReplaceTempView("dim_queue")

In [0]:
assignment_qualcoab_exceptions = spark.sql(f"""
                                                select distinct QUE_DESC
                                                from `{env}-cdh`.landed.params_assignment_qualcoab_exceptions
                                                where BusinessDate = '{Businessdate}'
                                            """)

assignment_qualcoab_exceptions.createOrReplaceTempView("assignment_qualcoab_exceptions")

In [0]:
dim_Collector = spark.sql(f"""
                        select distinct 
                                COL_ID
                              , COL_LOGIN
                        from `{env}-cdh`.landed.qca_dim_Collector
                        where BusinessDate = '{Businessdate}'
                    """)

dim_Collector.createOrReplaceTempView("dim_Collector")

In [0]:
assignment_queue_setup_qca = spark.sql(f"""
                                        select *
                                        from `d-cdh`.reference.params_assignment_queue_setup_qca
                                    """)

assignment_queue_setup_qca.createOrReplaceTempView("assignment_queue_setup_qca")

## Select Fields

#### Not Core Securitized

In [0]:
s1 = spark.sql(f"""    
                    select 's1' source, *
                    from (
                            select distinct 
                                  cast(cases.CASE_CODE as string) as CASE_CODE
                                , dcases.CASE_ACCOUNT_FOR_SEARCH
                                , cast(cases.CUST_CODE as string) as CUST_CODE
                                , CASE
                                    WHEN q.QUE_DESC ilike '%IRT%' THEN 'IRT'
                                    WHEN ext.EXTC_NAME ilike 'PRE%' THEN 'Outsourcing Collection' 
                                    WHEN EXT.EXTC_NAME = 'BRANCH_NETWORK' and q.QUE_DESC ilike '%NET' THEN 'FAs'
                                    WHEN q.QUE_DESC ilike 'ΔΤ3869%' AND EXC.QUE_DESC IS NULL THEN 'L3869 ΔΤ - FAs'
                                    WHEN q.QUE_DESC ilike '%ext' 
                                        AND EXT.EXTC_NAME NOT ilike 'PRE%' 
                                        AND EXT.EXTC_NAME IS NOT NULL 
                                        AND EXT.EXTC_NAME <> 'BRANCH_NETWORK' 
                                        AND EXT.EXTC_NAME NOT ilike 'N3869%' THEN 'Legal Agencies' 
                                    ELSE '-1'
                                  END   AS AssignmentChannel
                                , cast(cases.EXTC_CODE_PRIMARY as string) AS ExternalCompanyAssignment
                                , ext.EXTC_NAME			  AS ExternalCompanyName
                                , CASE 
                                    WHEN EXT.EXTC_NAME ilike 'PRE%' THEN EXT.EXTC_NAME 
                                    WHEN EXT.EXTC_NAME='BRANCH_NETWORK' and q.QUE_DESC ilike '%NET' THEN BR.BRN_DESC
                                    WHEN  q.QUE_DESC ilike 'ΔΤ3869%' AND EXC.QUE_DESC IS NULL  THEN BR.BRN_DESC
                                    WHEN q.QUE_DESC ilike '%ext'
                                        AND EXT.EXTC_NAME NOT ilike 'PRE%' 
                                        AND EXT.EXTC_NAME IS NOT NULL                           
                                        AND EXT.EXTC_NAME <> 'BRANCH_NETWORK'                 
                                        THEN EXT.EXTC_NAME                 
                                    ELSE '-1'
                                  END	AS AssignmentTeam
                                , CASE 
                                    WHEN EXT.EXTC_NAME='BRANCH_NETWORK' and q.QUE_DESC ilike '%NET' THEN cast(CASES.EMP_TAG as string)
                                    ELSE '-1'	
                                  END	AS AssignmentAgent
                                , col.COL_LOGIN           AS AssignmentAgentLoginName
                                , br.BRN_INTCODE	        AS Branch
                                , cast(null as string)    AS BranchDescription
                                , 'N/A'			              AS AssignmentProcess
                                , cases.SPET_CODE         AS portfolio_id_qca
                        from fs_cases_snap cases
                        inner join dim_cases_static dcases
                        on cases.case_code = dcases.case_code
                        left join dim_external_company ext
                        on ext.EXTC_ID = cases.EXTC_CODE_PRIMARY
                        left join dim_branch br
                        on cases.case_tag = br.BRN_ID
                        left join dim_queue q
                        on cases.QUE_CODE = q.QUE_ID
                        left join assignment_qualcoab_exceptions exc
                        on q.QUE_DESC = exc.QUE_DESC
                        left join dim_Collector col
                        on col.col_id = cases.emp_tag
                        where SPET_CODE NOT IN (1,2,18,20,22)
                    ) a 
                    where cast(a.AssignmentChannel as string) <> '-1'
                """)
                
s1.createOrReplaceTempView("s1")

#### Core Banking

In [0]:
s2 = spark.sql(f"""    
                    select 's2' source, *
                    from (
                        select distinct 
                                  cast(cases.CASE_CODE as string) as CASE_CODE
                                , dcases.CASE_ACCOUNT_FOR_SEARCH
                                , cast(cases.CUST_CODE as string) as CUST_CODE
                                , CASE
                                    WHEN (q.QUE_DESC ilike '%net') and (EXT.EXTC_NAME ilike 'BRANCH_NETWORK' or q.QUE_DESC  ilike 'ΔΤ3869_hub%') THEN 'HUB RMs'
                                    WHEN q.QUE_DESC ilike '%net_int' and EXT.EXTC_NAME ilike 'BRANCH_NETWORK' THEN 'HUB RMs / Call Center'
                                    WHEN (q.QUE_DESC ilike '%int' and q.QUE_DESC not ilike '%IRT%')
                                        or q.QUE_DESC  ilike 'IRT_Dou%'
                                        or LOWER(q.QUE_DESC) = 'sse84_loans'
                                        or q.QUE_DESC  = 'Private_Affluent_Customers'
                                        or q.QUE_DESC  = 'Settlements_Προσωρινών_Διαταγών_3869'
                                        or q.QUE_DESC  = 'Gefyra_Accounts'
                                        THEN 'Call Center'
                                    WHEN q.QUE_DESC  ilike '%int' and q.QUE_DESC ilike '%IRT%' THEN 'IRT'
                                    WHEN EXT.EXTC_NAME ilike 'PRE%' THEN 'Outsourcing Collection'
                                    WHEN q.QUE_DESC ilike '%ext'
                                        and EXT.EXTC_NAME NOT ilike 'PRE%'
                                        and EXT.EXTC_NAME <> 'BRANCH_NETWORK'
                                        and EXT.EXTC_NAME IS NOT NULL
                                        and EXT.EXTC_NAME in ('ANDRIKOPOULOS','GIANNINI','MOURGELAS','SIOUFAS','KORAKI','LEXIS','TSAFARAS')
                                        THEN 'Legal Agencies'
                                    WHEN EXT.EXTC_NAME ilike 'N3869%' THEN 'Legal Agencies 3869'
                                    ELSE '-1'
                                  END   AS AssignmentChannel
                                , cast(cases.EXTC_CODE_PRIMARY as string) AS ExternalCompanyAssignment
                                , EXT.EXTC_NAME			      AS ExternalCompanyName
                                , CASE
                                    WHEN EXT.EXTC_NAME ilike 'PRE%'	THEN EXT.EXTC_NAME
                                    WHEN ((q.QUE_DESC ilike '%net') and (EXT.EXTC_NAME ilike 'BRANCH_NETWORK' or q.QUE_DESC ilike 'ΔΤ3869_hub%'))
                                        or (q.QUE_DESC ilike '%net_int' and EXT.EXTC_NAME ilike 'BRANCH_NETWORK')
                                        THEN BR.BRN_DESC
                                    WHEN q.QUE_DESC ilike '%ext'
                                        and EXT.EXTC_NAME NOT ilike 'PRE%'
                                        and EXT.EXTC_NAME <> 'BRANCH_NETWORK'
                                        and EXT.EXTC_NAME IS NOT NULL
                                        and EXT.EXTC_NAME in ('ANDRIKOPOULOS','GIANNINI','MOURGELAS','SIOUFAS','KORAKI','LEXIS','TSAFARAS')
                                        THEN EXT.EXTC_NAME
                                    ELSE '-1'
                                  END   AS AssignmentTeam
                                , CASE 
                                    WHEN ((q.QUE_DESC ilike '%net') and (EXT.EXTC_NAME ilike 'BRANCH_NETWORK' or q.QUE_DESC ilike 'ΔΤ3869_hub%'))
                                      or (q.QUE_DESC ilike '%net_int' and EXT.EXTC_NAME ilike 'BRANCH_NETWORK')
                                      THEN  cast(CASES.EMP_TAG as string)
                                    ELSE '-1'	
                                  END   AS AssignmentAgent
                                , col.COL_LOGIN         AS AssignmentAgentLoginName
                                , br.BRN_INTCODE	      AS Branch
                                , cast(null as string)  AS BranchDescription
                                , 'N/A'			            AS AssignmentProcess
                                , cases.SPET_CODE       AS portfolio_id_qca
                        from fs_cases_snap cases
                        inner join dim_cases_static dcases
                        on cases.case_code = dcases.case_code
                        left join dim_external_company ext
                        on ext.EXTC_ID = cases.EXTC_CODE_PRIMARY
                        left join dim_branch br
                        on cases.case_tag = br.BRN_ID
                        left join dim_queue q
                        on cases.QUE_CODE = q.QUE_ID
                        left join assignment_qualcoab_exceptions exc
                        on q.QUE_DESC = exc.QUE_DESC
                        left join dim_Collector col
                        on col.col_id = cases.emp_tag
                        where SPET_CODE IN (1,2)
                    ) a 
                    where cast(a.AssignmentChannel as string) <> '-1'
                """)

s2.createOrReplaceTempView("s2")

#### Monza/Domus/Rhodium Retail

In [0]:
s3 = spark.sql(f""" 
                    select 's3' source, 
                              cast(cases.CASE_CODE as string) as CASE_CODE
                            , dcases.CASE_ACCOUNT_FOR_SEARCH
                            , cast(cases.CUST_CODE as string) as CUST_CODE	
                            , CASE
                                WHEN EXT.EXTC_NAME ilike 'PRE%' THEN 'Outsourcing Collection'
                                WHEN EXT.EXTC_NAME NOT ilike 'PRE%'
                                    AND EXT.EXTC_NAME IS NOT NULL
                                    AND EXT.EXTC_NAME <> 'BRANCH_NETWORK'
                                    AND EXT.EXTC_NAME NOT ilike 'N3869%'
                                    THEN 'Legal Agencies'
                                WHEN EXT.EXTC_NAME='BRANCH_NETWORK'
                                    AND EMP_TAG IS NOT NULL
                                    AND br.BRN_DESC ilike '%FAs%'
                                    AND br.BRN_DESC not ilike '%High%'
                                    THEN 'FAs'
                                WHEN EXT.EXTC_NAME='BRANCH_NETWORK'
                                    AND EMP_TAG IS NOT NULL
                                    AND br.BRN_DESC ilike '%High%'
                                    THEN 'FAs High Balance'
                                WHEN q.QUE_DESC ilike '%IRT%' THEN 'IRT'
                                ELSE NULL
                              END   AS AssignmentChannel
                            , CASE
                                WHEN EXT.EXTC_NAME='BRANCH_NETWORK' THEN NULL 
                                ELSE cast(cases.EXTC_CODE_PRIMARY as string)	
                              END	AS ExternalCompanyAssignment
                            , CASE
                                WHEN EXT.EXTC_NAME='BRANCH_NETWORK' THEN NULL 
                                ELSE EXT.EXTC_NAME	
                              END	AS ExternalCompanyName
                            , CASE
                                WHEN EXT.EXTC_NAME<>'BRANCH_NETWORK' THEN NULL 
                                ELSE br.BRN_DESC	
                              END	AS AssignmentTeam
                            , CASE
                                WHEN EXT.EXTC_NAME='BRANCH_NETWORK' THEN cast(CASES.EMP_TAG as string)
                                ELSE NULL
                              END	AS AssignmentAgent
                            , col.COL_LOGIN           AS AssignmentAgentLoginName
                            , br.BRN_INTCODE          AS Branch
                            , cast(null as string)    AS BranchDescription
                            , CASE 
                                WHEN EXT.EXTC_NAME ilike 'PRE%' THEN 'Collection'
                                WHEN EXT.EXTC_NAME NOT ilike 'PRE%'     
                                    AND EXT.EXTC_NAME IS NOT NULL                           
                                    AND EXT.EXTC_NAME <> 'BRANCH_NETWORK'                 
                                    AND EXT.EXTC_NAME NOT ilike 'N3869%'					  
                                    THEN 'LF_Offer'
                                WHEN EXT.EXTC_NAME='BRANCH_NETWORK'
                                    AND EMP_TAG IS NOT NULL
                                    AND br.BRN_DESC ilike '%FAs%'
                                    AND br.BRN_DESC not ilike '%High%' 
                                    THEN 'CA/FA/Sme Officer'
                                WHEN EXT.EXTC_NAME='BRANCH_NETWORK' 
                                    AND EMP_TAG IS NOT NULL 
                                    AND br.BRN_DESC ilike '%High%' 
                                    THEN 'CA/FA/Sme Officer'
                                WHEN q.QUE_DESC ilike '%IRT%' THEN 'IRT'
                                WHEN exc.assignment_mapping IS NOT NULL THEN exc.assignment_mapping
                                ELSE 'Rest'   
                              END   AS AssignmentProcess  
                            , cases.SPET_CODE       AS portfolio_id_qca
                    from fs_cases_snap cases
                    inner join dim_cases_static dcases
                    on cases.case_code = dcases.case_code
                    and dcases.CASE_CODE = dcases.CASE_CODE_PRIMARY 
                    left join dim_external_company ext
                    on ext.EXTC_ID = cases.EXTC_CODE_PRIMARY
                    left join dim_branch br
                    on cases.case_tag = br.BRN_ID
                    left join dim_queue q
                    on cases.QUE_CODE = q.QUE_ID
                    left join assignment_queue_setup_qca exc
                    on q.QUE_DESC = exc.queue_desc
                    left join dim_Collector col
                    on col.col_id = cases.emp_tag
                    where SPET_CODE in (18,20,22)
                """)

s3.createOrReplaceTempView("s3")

## Create the Final Table

In [0]:
combined_df = s1.unionByName(s2)\
                .unionByName(s3)\
                .dropDuplicates()

combined_df.createOrReplaceTempView("combined_df")

In [0]:
combined_df = combined_df.withColumn("BusinessDate", lit(Businessdate))
combined_df = combined_df.withColumn("LoadingDateTime", lit(current_timestamp()))

combined_df.createOrReplaceTempView("combined_df")

## Compare Tables

#### Extract Original

In [0]:
case_assignment= spark.sql(f"""
                            select *
                            from `{env}-cdh`.landed.qca_case_assignment
                            where BusinessDate = '{Businessdate}'
                        """)

case_assignment.createOrReplaceTempView("case_assignment")

#### Define common cols

In [0]:
# Assume df1 and df2 are your two DataFrames
df1 = combined_df
df2 = case_assignment

# Step 1: Identify common columns dynamically
common_cols = list(set(df1.columns) & set(df2.columns))
common_cols.remove("LoadingDateTime")

print("Common columns being compared:", common_cols)

#### Make the Compares

In [0]:
# Step 2: Normalize values to avoid mismatches
df1_common = df1.select([lower(col(c)).alias(c) for c in common_cols]).dropDuplicates()
df2_common = df2.select([lower(col(c)).alias(c) for c in common_cols]).dropDuplicates()

# Check row counts
print("df1_common rows:", df1_common.count())
print("df2_common rows:", df2_common.count())

# Step 3: Compare
diff1 = df1_common.exceptAll(df2_common)
diff2 = df2_common.exceptAll(df1_common)

if diff1.isEmpty() and diff2.isEmpty():
    print("DataFrames are equal in common columns.")
else:
    print("Differences found:")
    display(diff1)
    display(diff2)


## Important

In [0]:
# It is expected to appear differences because in on-prem implementation there was a date-missmatch on data of DWH.adm.galaxycoreperimeter
# So in this implementation the error is fixed 